# Fase 1: Preprocessing & Pembersihan Dataset

Notebook ini adalah **Fase 1** dari pipeline pelatihan model deteksi tulisan AI.
Tugas utama notebook ini:
1. **Memuat Data**: Membaca raw dataset `model_training_dataset.csv` (dengan opsi *sampling* untuk mengurangi beban memori).
2. **Transformasi Format (Melt)**: Mengubah kolom `human_text` dan `ai_text` menjadi format *long-form* (`text`, `source`).
3. **Pembersihan Teks & Placeholder**: Menangani token `generic_name` dll menggunakan metode *Natural Substitution* agar konteks kalimat tetap terjaga.
4. **Deteksi Bahasa**: Menambahkan label bahasa (Inggris/Indonesia) untuk setiap baris.
5. **Ekspor Checkpoint**: Menyimpan hasil akhirnya dalam format `.parquet` berkinerja tinggi untuk diteruskan ke Fase 2.


In [1]:
# Instalasi dependensi jika belum ada
import sys
import subprocess

required = ["pandas", "numpy", "ipywidgets", "pyarrow", "fastparquet"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Memasang {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import re
import gc
import time

# Konfigurasi Pandas
pd.set_option('display.max_colwidth', 200)
print("Library berhasil diimpor.")


Library berhasil diimpor.


## 1. Pemuatan Data (Sampling)
Dataset lengkap berukuran ~3.7 GB. Di sini Anda dapat memuat seluruh data atau hanya sebagian (*sample*) untuk prototipe cepat.


In [2]:
DATA_PATH = "model_training_dataset.csv"
df_raw = None

# Widget UI
sample_dropdown = widgets.Dropdown(
    options=[('10.000 baris (Quick Test)', 10000), 
             ('50.000 baris (Prototype)', 50000), 
             ('200.000 baris (Medium)', 200000), 
             ('Semua Baris (Full)', None)],
    value=50000,
    description='Jumlah Data:',
    style={'description_width': 'initial'}
)
load_btn = widgets.Button(description="Muat Dataset", button_style='primary', icon='download')
out_load = widgets.Output()

def load_data(b):
    global df_raw
    with out_load:
        clear_output()
        print("Membaca file CSV... mohon tunggu...")
        try:
            start_time = time.time()
            if sample_dropdown.value is None:
                df_raw = pd.read_csv(DATA_PATH)
            else:
                df_raw = pd.read_csv(DATA_PATH, nrows=sample_dropdown.value)
            
            elapsed = time.time() - start_time
            print(f"Berhasil memuat {len(df_raw):,} baris dalam {elapsed:.2f} detik.")
            display(df_raw.head(2))
            
            # Cek memory usage
            mem_mb = df_raw.memory_usage(deep=True).sum() / 1024**2
            print(f"Estimasi Memori: {mem_mb:.2f} MB")
            
        except FileNotFoundError:
            print(f"File tidak ditemukan di: {DATA_PATH}")

load_btn.on_click(load_data)
display(widgets.HBox([sample_dropdown, load_btn]), out_load)


Output()

## 2. Transformasi Format (Melt)
Mengubah struktur *wide-format* (id, human_text, ai_text) menjadi *long-format* agar mudah diproses oleh model klasifikasi.
Kolom baru: `text` dan `source` (berisi 'human' atau 'ai').


In [7]:
df_data = None

def melt_dataframe(df):
    print("Memulai proses transformasi (Melt)...")
    
    # Pisahkan bagian human dan ai
    df_human = df[['id', 'human_text', 'instructions']].rename(columns={'human_text': 'text'})
    df_human['source'] = 'human'
    
    df_ai = df[['id', 'ai_text', 'instructions']].rename(columns={'ai_text': 'text'})
    df_ai['source'] = 'ai'
    
    # Gabungkan
    df_long = pd.concat([df_human, df_ai], ignore_index=True)
    
    # Drop N/A dan string kosong
    df_long = df_long.dropna(subset=['text'])
    df_long = df_long[df_long['text'].str.strip() != '']
    
    # PENTING: Pastikan kolom 'id' bertipe string agar konsisten antar notebook
    # Notebook 4 (Training) melakukan group-split berdasarkan 'id' ini.
    # Inkonsistensi tipe (int vs str) akan menyebabkan merge gagal.
    df_long['id'] = df_long['id'].astype(str)
    
    # Shuffle untuk memecah urutan tapi 'id' TETAP ada, tidak di-drop.
    # Ini penting agar Notebook 4 bisa lakukan group-based split
    # (split berdasarkan ID unik agar pasangan human+AI tidak terpisah split).
    df_long = df_long.sample(frac=1.0, random_state=42).reset_index(drop=True)
    
    print(f"Transformasi selesai. Total baris: {len(df_long):,}")
    print(f"ID unik: {df_long['id'].nunique():,} (setiap ID punya 1 versi human + 1 versi AI)")
    return df_long

# Eksekusi (jika df_raw sudah ada)
if df_raw is not None:
    df_data = melt_dataframe(df_raw)
    display(df_data[['id', 'source']].groupby('source')['id'].count().rename("Jumlah Baris").to_frame())
    display(df_data.head(4))
else:
    print("Silakan muat dataset di langkah sebelumnya terlebih dahulu.")


Memulai proses transformasi (Melt)...
Transformasi selesai. Total baris: 100,000
ID unik: 50,000 (setiap ID punya 1 versi human + 1 versi AI)


,Jumlah Baris
source,
ai,50000
human,50000


,id,text,instructions,source
0,317fc428-de31-4bec-a35e-41688ec60018,"Having looked into family records, I believe I have a connection to the area--so I'm looking forward to exploring the city, visiting the places my ancestors may have lived and experienced, and fin...","Task: \n\n- Research LOCATION_NAME and the attractions, events, restaurants that can be found in this location.\n- Plan a trip for a duration of about one week, including the places that you want ...",ai
1,7bedae35-f300-40e7-8b90-1d2647a45f19,"Similarly, in relationships, mistakes are inevitable—but rather than avoiding them, use mistakes as a chance to grow and learn how to be a better friend.\n\nI agree with this quote as it encourage...","Task: \n\n1. Analyze the quote ""A problem is a chance for you to do your best"" and explain what it means.\n2. Provide examples of how exhibiting this attitude in action can help in school, in spor...",ai
2,9f2a29ab-97d4-414d-ab74-f9a0d941f575,"You also get more noticed, and people will like to be around you, because you're more active, and you're more unique than most people.. No one can stop you, because your mindset is on a whole new ...","Task: Research the benefits of being an action taker, rather than a dreamer; identify three reasons why one should take action rather than remain inactive; provide examples for each reason.",human
3,b2f04403-b81a-41b1-ae58-c1261bd0b18a,Data has shown that students who spend more time on their cell phones in the classroom tend to have lower grades and decreased engagement in class activities. In order to make the use of cell phon...,Task: Research how the use of cell phones in the classroom affects academic performance and engagement. Investigate whether there is a correlation between the amount of screen time and educational...,ai


## 3. Pembersihan & Substitusi Placeholder

Dataset ini diketahui memiliki *placeholders* seperti `Generic_Name`, `Generic_City`.
Untuk model berbasis *embedding* (seperti USE-M), kalimat berbunyi: *"My name is Generic_Name and I live in Generic_City"* akan merusak pemahaman semantik. 
Solusi terbaik adalah **Natural Substitution**: Mengganti placeholder tersebut dengan kata standar yang tidak merusak grammar.

Selain itu, teks akan dibersihkan dari spasi berlebih dan dilakukan *language detection* (deteksi bahasa) sederhana menggunakan heuristik *stop words*.


In [8]:
# 1. Kamus Substitusi Natural
SUBSTITUTIONS = {
    r'(?i)generic_name': 'Alex',
    r'(?i)generic_namehad': 'Alex had',
    r'(?i)generic_city': 'London',
    r'(?i)generic_citynbsp': 'London',
    r'(?i)generic_school': 'University'
}

# 2. Heuristik Deteksi Bahasa Sederhana
# Karena dataset asli 99% bahasa Inggris, kita hanya perlu heuristik ringan.
EN_STOPWORDS = {'and', 'the', 'is', 'in', 'of', 'to', 'this', 'that', 'it', 'for'}
ID_STOPWORDS = {'dan', 'yang', 'di', 'dari', 'ke', 'ini', 'itu', 'pada', 'dengan', 'untuk'}

def detect_language(text):
    text_lower = text.lower()
    words = set(re.findall(r'\b\w+\b', text_lower))
    
    id_score = len(words.intersection(ID_STOPWORDS))
    en_score = len(words.intersection(EN_STOPWORDS))
    
    if id_score > en_score and id_score > 0:
        return 'id'
    else:
        return 'en' # Default fallback ke EN

# 3. Fungsi Pembersihan Utama
def clean_text(text, method='natural'):
    if not isinstance(text, str):
        return ""
        
    # Unicode Normalization & strip
    text = text.replace('\r', '').strip()
    
    # Handle Placeholders
    if method == 'natural':
        for pattern, replacement in SUBSTITUTIONS.items():
            text = re.sub(pattern, replacement, text)
    elif method == 'redact':
        text = re.sub(r'(?i)\bgeneric_\w+\b', '[REDACTED]', text)
        
    # Rapikan multiple spaces
    text = re.sub(r' +', ' ', text)
    return text


In [9]:
clean_dropdown = widgets.Dropdown(
    options=[('Natural Substitution (Rekomendasi)', 'natural'), 
             ('Ganti dengan [REDACTED]', 'redact'), 
             ('Biarkan apa adanya', 'none')],
    value='natural',
    description='Strategi:',
)
clean_btn = widgets.Button(description="Mulai Pembersihan", button_style='success', icon='magic')
out_clean = widgets.Output()

def process_cleaning(b):
    with out_clean:
        clear_output()
        if df_data is None:
            print("Error: DataFrame `df_data` belum tersedia!")
            return
            
        print("Menerapkan pembersihan teks...")
        start_time = time.time()
        
        # Eksekusi apply (bisa memakan waktu untuk dataset besar)
        if clean_dropdown.value != 'none':
            df_data['text_clean'] = df_data['text'].apply(lambda x: clean_text(x, method=clean_dropdown.value))
        else:
            df_data['text_clean'] = df_data['text']
            
        print("Mendeteksi bahasa...")
        df_data['language'] = df_data['text_clean'].apply(detect_language)
        
        elapsed = time.time() - start_time
        print(f"Selesai dalam {elapsed:.2f} detik!")
        
        display(df_data['language'].value_counts().to_frame("Distribusi Bahasa"))
        
        # Tampilkan perbandingan
        sample = df_data[df_data['text'].str.contains('Generic_', case=False, na=False)].head(2)
        if len(sample) > 0:
            print("\nContoh perubahan (Placeholder):")
            for _, row in sample.iterrows():
                print(f"ORIGINAL: {row['text'][:150]}...")
                print(f"CLEANED : {row['text_clean'][:150]}...")
                print("-" * 50)

clean_btn.on_click(process_cleaning)
display(widgets.HBox([clean_dropdown, clean_btn]), out_clean)


Output()

## 4. Simpan Checkpoint (Parquet)
Simpan hasil akhir (*cleaned text*, *source*, *language*) ke dalam format `.parquet` yang sangat efisien dan menyimpan tipe data secara persisten. File ini akan menjadi input untuk Notebook Fase 2.


In [11]:
import os
save_btn = widgets.Button(description="Simpan ke Parquet", button_style='info', icon='save')
out_save = widgets.Output()

def save_checkpoint(b):
    with out_save:
        clear_output()
        if 'text_clean' not in df_data.columns:
            print("Anda harus menjalankan proses pembersihan terlebih dahulu!")
            return
            
        output_file = "data_cleaned.parquet"
        print(f"Menyimpan ke {output_file}...")
        
        # Kita hanya menyimpan kolom yang diperlukan untuk fase selanjutnya
        cols_to_save = ['id', 'text_clean', 'source', 'instructions', 'language']
        df_save = df_data[[c for c in cols_to_save if c in df_data.columns]]
        
        df_save.to_parquet(output_file, engine='fastparquet', index=False)
        print(f"Checkpoint berhasil disimpan! Ukuran file: {os.path.getsize(output_file) / 1024**2:.2f} MB")
        print("\nAnda siap melanjutkan ke Notebook 2: 02_feature_engineering.ipynb")

save_btn.on_click(save_checkpoint)
display(save_btn, out_save)


Button(button_style='info', description='Simpan ke Parquet', icon='save', style=ButtonStyle())

Output()